
# GDPR Legality Classification with **PDF Attachment** (gpt-4o-mini)

This notebook uploads your local **GDPR.pdf** to the OpenAI Files API and **attaches the file** to each model request.  
The model must judge each case **STRICTLY grounded** on the attached PDF.

**Pipeline**  
1. Load cases from JSONL  
2. Upload `GDPR.pdf` → get `file_id`  
3. For each case, call **Responses API** with two content parts:  
   - `input_text`: the case content  
   - `input_file`: the uploaded `GDPR.pdf`  
4. Model outputs one of: `LEGAL` or `ILLEGAL` (single token word)  
5. Ground truth = `violated_articles` is empty ⇒ `LEGAL`, else `ILLEGAL`  
6. Report accuracy and save a CSV


In [4]:

# If needed, install deps:
# %pip install python-dotenv openai tqdm pandas


In [5]:

import os
import json
from typing import List, Dict, Any
from dataclasses import dataclass
from dotenv import load_dotenv
from tqdm import tqdm
import pandas as pd
import time
import re

from openai import OpenAI

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# ---- Paths ----
JSONL_PATH = "/Users/taeyoonkwack/Documents/PrivaCI-Bench/HF_cache/cases/HIPAA/balanced_sample.jsonl"
GDPR_PDF_PATH = "/Users/taeyoonkwack/Documents/PrivaCI-Bench/HF_cache/KBs/HIPAA/HIPAA.pdf"
OUTPUT_CSV = "hipaa_legality_predictions_attached_pdf.csv"

print("JSONL_PATH:", JSONL_PATH)
print("hipaa_PDF_PATH:", GDPR_PDF_PATH)


JSONL_PATH: /Users/taeyoonkwack/Documents/PrivaCI-Bench/HF_cache/cases/HIPAA/balanced_sample.jsonl
hipaa_PDF_PATH: /Users/taeyoonkwack/Documents/PrivaCI-Bench/HF_cache/KBs/HIPAA/HIPAA.pdf


In [6]:

def load_cases(jsonl_path: str) -> List[Dict[str, Any]]:
    rows = []
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            line=line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows

def ground_truth_label(violated_articles: List[str]) -> str:
    return "LEGAL" if not violated_articles else "ILLEGAL"

def coerce_label(text: str) -> str:
    t = (text or "").strip().upper()
    # keep only letters to be safe
    t = re.sub(r"[^A-Z]", "", t)
    if "ILLEGAL" in t:
        return "ILLEGAL"
    if "LEGAL" in t:
        return "LEGAL"
    # final fallback
    return "LEGAL"


In [7]:

# 1) Load cases
cases = load_cases(JSONL_PATH)
print(f"Loaded {len(cases)} cases")

# 2) Upload GDPR.pdf
print("Uploading GDPR.pdf to Files API ...")
file_obj = client.files.create(
    file=open(GDPR_PDF_PATH, "rb"),
    purpose="assistants"  # also works with "responses" on recent SDKs; "assistants" remains accepted
)
GDPR_FILE_ID = file_obj.id
print("Uploaded. file_id =", GDPR_FILE_ID)


Loaded 95 cases
Uploading GDPR.pdf to Files API ...
Uploaded. file_id = file-8oyNCN8dVRXAgDRhqCij2A


In [8]:

# 3) Inference loop using Responses API with input_file attachment

SYSTEM_PROMPT = (
    "You are a ACLU compliance analyst. "
    "Using ONLY the attached ACLU PDF as the source of truth, "
    "decide whether the described data processing is LEGAL or ILLEGAL under the ACLU. "
    "Respond with a single word exactly: LEGAL or ILLEGAL."
)

def classify_case(case_content: str, file_id: str, max_retries: int = 3, sleep_s: float = 2.0) -> str:
    for attempt in range(1, max_retries+1):
        try:
            resp = client.responses.create(
                model="gpt-4o-mini",
                temperature=0,
                input=[
                    {
                        "role": "system",
                        "content": [
                            {"type": "input_text", "text": SYSTEM_PROMPT}
                        ]
                    },
                    {
                        "role": "user",
                        "content": [
                            {"type": "input_text", "text": f"[CASE]\n{case_content}\n\nTASK: Output exactly one word: LEGAL or ILLEGAL."},
                            {"type": "input_file", "file_id": file_id}
                        ]
                    }
                ]
            )
            # Try friendly accessors first
            out = getattr(resp, "output_text", None)
            if not out:
                # Fallback: walk the JSON-like structure
                out = ""
                try:
                    for item in resp.output or []:
                        for c in getattr(item, "content", []) or []:
                            if getattr(c, "type", "") == "output_text":
                                out += getattr(c, "text", "") + " "
                except Exception:
                    pass
            label = coerce_label(out)
            return label
        except Exception as e:
            if attempt == max_retries:
                print(f"[ERROR] API call failed after {max_retries} attempts: {e}")
                return "LEGAL"  # neutral fallback
            time.sleep(sleep_s)

rows = []
preds, truth = [], []

for idx, case in enumerate(tqdm(cases, desc="Classifying")):
    case_content = case.get("case_content","")
    violated_articles = case.get("violated_articles", []) or []
    gt = ground_truth_label(violated_articles)

    yhat = classify_case(case_content, GDPR_FILE_ID)

    preds.append(yhat); truth.append(gt)
    rows.append({
        "index": idx,
        "prediction": yhat,
        "ground_truth": gt,
        "is_correct": yhat == gt,
        "violated_articles": "; ".join(violated_articles),
        "case_excerpt": case_content[:300].replace("\n"," ") + ("..." if len(case_content)>300 else ""),
    })

correct = sum(1 for p,t in zip(preds, truth) if p == t)
total = len(truth) if truth else 1
acc = correct/total
print(f"\nAccuracy: {acc:.4f} ({correct}/{total})")

df = pd.DataFrame(rows)
df.to_csv(OUTPUT_CSV, index=False)
print("Saved ->", OUTPUT_CSV)


Classifying: 100%|██████████| 95/95 [06:39<00:00,  4.21s/it]


Accuracy: 0.3579 (34/95)
Saved -> hipaa_legality_predictions_attached_pdf.csv


In [9]:

import caas_jupyter_tools as cj, pandas as pd
df = pd.read_csv("gdpr_legality_predictions_attached_pdf.csv")
cj.display_dataframe_to_user("GDPR Legality with Attached PDF", df)


ModuleNotFoundError: No module named 'caas_jupyter_tools'